# Online Retail Customer Segmentation

**Recruiter-facing end-to-end analysis · Unsupervised RFM segmentation · Python 3.12/3.13**

> The pipeline segments 4,338 customers into 2 stable RFM groups (silhouette 0.432, seed ARI 0.999).

## Executive summary

**Objective:** Create auditable customer profiles after explicit cancellation, return, duplicate, and missing-ID treatment.

**Data:** 541,909 UCI Online Retail transaction rows from December 2010 through December 2011.

**Verified result:** The pipeline segments 4,338 customers into 2 stable RFM groups (silhouette 0.432, seed ARI 0.999).

**Decision supported:** Design differentiated retention and reactivation experiments without sensitive profiling.

The figures, tables, metrics, and execution counts in this notebook are saved outputs from the bundled data.

## 1. Business understanding

**Primary user:** An e-commerce CRM or retention analyst.

**Decision:** Design differentiated retention and reactivation experiments without sensitive profiling.

**Why it matters:** A technically accurate result is useful only when its error costs, uncertainty, and decision boundary are visible. This project stays within the evidence available in the source data.

## 2. Analytical objective and success criteria

Technical success requires portable execution, explicit data-quality evidence, a justified baseline, leakage-safe validation, task-appropriate metrics, diagnostics, and saved artifacts. Business success requires a specific recommendation supported by the observed result without invented financial impact.

## 3. Reproducible environment

In [1]:
from pathlib import Path
import hashlib, importlib.util, json, os, platform, tempfile, time
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio-matplotlib-cache"))
import matplotlib, numpy as np, pandas as pd, scipy, sklearn
SLUG = '12-online-retail-segmentation'
def locate_project():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = base if base.name == SLUG else base / "projects" / SLUG
        if (candidate / "src" / "analysis.py").is_file(): return candidate
    raise FileNotFoundError(SLUG)
PROJECT_ROOT = locate_project(); DATA_DIR = PROJECT_ROOT / "data"; REPORTS_DIR = PROJECT_ROOT / "reports"
print(pd.Series({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__, "SciPy": scipy.__version__, "scikit-learn": sklearn.__version__, "Matplotlib": matplotlib.__version__}, name="version").to_string())
print(f"\nProject: {PROJECT_ROOT.name}")

Python          3.12.13
pandas            2.2.3
NumPy             2.3.5
SciPy            1.17.0
scikit-learn      1.8.0
Matplotlib       3.10.8

Project: 12-online-retail-segmentation


## 4. Data provenance and scope

UCI Online Retail dataset; preserve UCI attribution and verify source-specific reuse terms before redistribution.

The next cells expose exact files, byte sizes, checksums, schemas, and sample records.

### 4.1 Source-file inventory

In [2]:
rows=[]
for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and path.name != "README.md": rows.append({"file": path.name, "size_mb": round(path.stat().st_size/1_000_000,3), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]})
inventory=pd.DataFrame(rows); print(inventory.to_string(index=False))

              file  size_mb           sha256
online_retail.xlsx   23.715 43465a06f2ccf7c8


### 4.2 Raw-record and schema preview

In [3]:
def preview(path):
    if path.suffix.lower()==".csv": return pd.read_csv(path, nrows=5)
    if path.suffix.lower()==".xlsx": return pd.read_excel(path, nrows=5)
    return None
for path in sorted(DATA_DIR.iterdir()):
    frame=preview(path)
    if frame is None: continue
    for column in frame.select_dtypes(include="object"): frame[column]=frame[column].astype(str).str.replace(r"\\s+"," ",regex=True).str.slice(0,100)
    print(f"\n{path.name}: {frame.shape[1]} columns"); print(frame.to_string(index=False,max_cols=12))


online_retail.xlsx: 8 columns
 InvoiceNo StockCode                         Description  Quantity         InvoiceDate  UnitPrice  CustomerID        Country
    536365    85123A  WHITE HANGING HEART T-LIGHT HOLDER         6 2010-12-01 08:26:00       2.55       17850 United Kingdom
    536365     71053                 WHITE METAL LANTERN         6 2010-12-01 08:26:00       3.39       17850 United Kingdom
    536365    84406B      CREAM CUPID HEARTS COAT HANGER         8 2010-12-01 08:26:00       2.75       17850 United Kingdom
    536365    84029G KNITTED UNION FLAG HOT WATER BOTTLE         6 2010-12-01 08:26:00       3.39       17850 United Kingdom
    536365    84029E      RED WOOLLY HOTTIE WHITE HEART.         6 2010-12-01 08:26:00       3.39       17850 United Kingdom


## 5. Data-quality assessment

The pipeline checks missingness, duplicates, invalid fields, identifiers, cardinality, and problem-specific leakage or chronology risks. No row is silently removed.

## 6. Reusable implementation

Large functions are kept in source code so the notebook remains a readable analytical narrative.

In [4]:
source_path=PROJECT_ROOT/"src"/"analysis.py"; text=source_path.read_text(encoding="utf-8")
print(f"Reusable implementation: {len(text.splitlines())} lines")
print("Functions:", ", ".join(line.split("(")[0].replace("def ","").strip() for line in text.splitlines() if line.startswith("def ")))

Reusable implementation: 146 lines
Functions: _segment_name, run_analysis


## 7. Methodology and hypotheses

Transaction cleaning, customer-level RFM, log scaling, K-means/Gaussian-mixture comparison, silhouette/Davies–Bouldin/Calinski–Harabasz metrics, seed stability, outlier sensitivity, PCA, and evidence-based naming.

The central hypothesis is that the audited features or group structure contain decision-relevant signal beyond the documented baseline. Exploratory findings are not presented as causal effects.

## 8. Execute the complete pipeline

This cell reruns cleaning, feature engineering, model/statistical analysis, validation, tables, figures, and model artifacts.

In [5]:
spec=importlib.util.spec_from_file_location("rebuilt_12_online_retail_segmentation", PROJECT_ROOT/"src"/"analysis.py")
analysis=importlib.util.module_from_spec(spec); spec.loader.exec_module(analysis)
started=time.perf_counter(); results=analysis.run_analysis(); runtime=time.perf_counter()-started
assert results["status"]=="passed"
print(f"Pipeline status: {results['status']}\nRuntime: {runtime:.2f} seconds")

Pipeline status: passed
Runtime: 39.56 seconds


## 9. Executed data-quality evidence

In [6]:
for path in sorted((REPORTS_DIR/"tables").glob("*data_quality.csv")):
    frame=pd.read_csv(path); print(f"\n{path.name} ({len(frame)} fields)"); print(frame.to_string(index=False,max_rows=30))


data_quality.csv (8 fields)
     column          dtype  missing_count  missing_percent  unique_values  constant
  InvoiceNo         object              0            0.000          25900     False
  StockCode         object              0            0.000           4070     False
Description         object           1454            0.268           4224     False
   Quantity          int64              0            0.000            722     False
InvoiceDate datetime64[ns]              0            0.000          23260     False
  UnitPrice        float64              0            0.000           1630     False
 CustomerID        float64         135080           24.927           4373     False
    Country         object              0            0.000             38     False


## 10. Baseline, candidates, and primary result

In [7]:
primary=REPORTS_DIR/"tables"/'cluster_model_selection.csv'
frame=pd.read_csv(primary); print(f"Primary evidence: {primary.name}, shape={frame.shape}")
print(frame.head(15).round(4).to_string(index=False))
print("\nVerified result:\n" + 'The pipeline segments 4,338 customers into 2 stable RFM groups (silhouette 0.432, seed ARI 0.999).')

Primary evidence: cluster_model_selection.csv, shape=(12, 7)
          method  clusters  silhouette  davies_bouldin  calinski_harabasz  mean_seed_stability_ari  smallest_cluster_share
          kmeans         2      0.4323          0.8925          4367.3231                   0.9991                  0.3840
gaussian_mixture         2      0.2889          1.0670          2288.4854                      NaN                  0.3442
          kmeans         3      0.3370          1.0483          3625.3034                   0.9966                  0.1773
gaussian_mixture         3      0.2596          1.2165          2795.2269                      NaN                  0.2026
          kmeans         4      0.3372          1.0106          3328.9191                   0.9697                  0.1634
gaussian_mixture         4      0.1728          1.7220          2148.4594                      NaN                  0.1925
          kmeans         5      0.3163          0.9878          3192.9771     

## 11. Validation, diagnostics, and robustness

In [8]:
sections=[key for key in ["validation","model_selection","tuning","residual_diagnostics","outlier_sensitivity","participant_bootstrap_intervals","diagnostic_90_percent_interval","empirical_90_percent_interval"] if key in results]
for key in sections: print(f"\n{key.upper()}\n"+json.dumps(results[key],indent=2)[:6000])


MODEL_SELECTION
{
  "methods": [
    "K-means",
    "Gaussian mixture"
  ],
  "candidate_clusters": [
    2,
    3,
    4,
    5,
    6,
    7
  ],
  "selection_rule": "composite rank among K-means candidates with >=5% smallest segment and >=0.80 seed ARI",
  "selected_clusters": 2,
  "selected_silhouette": 0.4323284246768482,
  "selected_davies_bouldin": 0.8924990931024944,
  "selected_calinski_harabasz": 4367.323097619211,
  "selected_seed_stability_ari": 0.9990752738031528
}

OUTLIER_SENSITIVITY
{
  "winsorized_at_99_percent_ari_vs_primary": 0.9852570608898947
}


## 12. Visual evidence

### Online Retail Segmentation Evidence

![online_retail_segmentation_evidence](../reports/figures/online_retail_segmentation_evidence.png)

### Rfm Cluster Projection

![rfm_cluster_projection](../reports/figures/rfm_cluster_projection.png)

## 13. Business interpretation

The pipeline segments 4,338 customers into 2 stable RFM groups (silhouette 0.432, seed ARI 0.999).

The correct action is to use this result as evidence for **Design differentiated retention and reactivation experiments without sensitive profiling.**, while retaining the documented baseline and monitoring the error or sensitivity segments.

## 14. Prioritized recommendations

1. Use the verified result to define a controlled follow-up rather than an automatic decision.
2. Monitor the weakest subgroup, time window, interval coverage, or cluster sensitivity shown in the saved tables.
3. Revalidate against a transparent baseline whenever the data or operating context changes.

## 15. Limitations, ethics, and responsible use

Segments describe historical purchasing behavior and must not support discriminatory pricing, credit decisions, or sensitive profiling.

Automated outputs remain associative unless a causal study design says otherwise.

## 16. Saved-artifact integrity

In [9]:
rows=[]
for path in sorted(REPORTS_DIR.rglob("*")):
    if path.is_file(): rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_kb": round(path.stat().st_size/1000,1), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:12]})
artifacts=pd.DataFrame(rows); print(artifacts.to_string(index=False,max_rows=80))

                                               artifact  size_kb       sha256
reports/figures/online_retail_segmentation_evidence.png    490.0 673a620498aa
             reports/figures/rfm_cluster_projection.png    370.5 894b66377586
                                   reports/metrics.json      1.9 f92d38c993f6
                     reports/tables/cleaning_funnel.csv      0.2 bb28379561ba
             reports/tables/cluster_model_selection.csv      1.3 a47625b795df
                   reports/tables/customer_segments.csv    231.2 a3b7faa72b33
                        reports/tables/data_quality.csv      0.4 a75bf6347923
                    reports/tables/segment_profiles.csv      0.3 0b23d91ba070


## 17. Acceptance check

In [10]:
metrics=json.loads((REPORTS_DIR/"metrics.json").read_text(encoding="utf-8"))
assert metrics["status"]=="passed"
assert list((REPORTS_DIR/"figures").glob("*.png"))
assert list((REPORTS_DIR/"tables").glob("*.csv"))
assert all(path.stat().st_size>0 for path in REPORTS_DIR.rglob("*") if path.is_file())
print("PASS: metrics status, figures, tables, and non-empty artifacts verified")

PASS: metrics status, figures, tables, and non-empty artifacts verified


## 18. Conclusion

The project addressed create auditable customer profiles after explicit cancellation, return, duplicate, and missing-id treatment. using transaction cleaning, customer-level rfm, log scaling, k-means/gaussian-mixture comparison, silhouette/davies–bouldin/calinski–harabasz metrics, seed stability, outlier sensitivity, pca, and evidence-based naming. The final verified conclusion is: **The pipeline segments 4,338 customers into 2 stable RFM groups (silhouette 0.432, seed ARI 0.999).** The next responsible step is external or current-data validation before operational use.

## 19. Reproduce locally

```bash
python projects/12-online-retail-segmentation/src/analysis.py
python scripts/execute_notebooks.py --project 12-online-retail-segmentation
```